## Instant Indexing For Blogger — fixed (uses google-auth, not the deprecated oauth2client)

In [ ]:
#@title Connect Your Google Drive
Connect_Google_Drive = "Yes" #@param ["Yes", "No"]
if Connect_Google_Drive == "Yes":
  from google.colab import drive
  drive.mount('/content/drive')
  print("Successfully Connected!")
else:
  print("Not Connected!")

In [ ]:
#@title Provide Complete Path Of The API Key Here
API_Path = "/content/drive/MyDrive/API_Key.json" #@param {type:"string"}

!pip install -q google-auth requests

from google.oauth2 import service_account
from google.auth.transport.requests import AuthorizedSession

SCOPES = ["https://www.googleapis.com/auth/indexing", "https://www.googleapis.com/auth/webmasters.readonly"]
ENDPOINT = "https://indexing.googleapis.com/v3/urlNotifications:publish"
print('*'*50);print("Scopes & Endpoint Configured...");print('*'*50);print("Adding Key...");print('*'*50);

credentials = service_account.Credentials.from_service_account_file(API_Path, scopes=SCOPES)
http = AuthorizedSession(credentials)
print("Credentials Successfully Authorized!");print('*'*50);

In [ ]:
#@title Step 1 — Check If It's Already Indexed
#@markdown siteUrl must exactly match how the property is added in Search Console (e.g. https://freestackhub.blogspot.com/)
siteUrl = "" #@param {type:"string"}
urlToCheck = "" #@param {type:"string"}

INSPECT_ENDPOINT = "https://searchconsole.googleapis.com/v1/urlInspection/index:inspect"
print('*'*50);print("Checking index status...");print('*'*50);

resp = http.post(INSPECT_ENDPOINT, json={'inspectionUrl': urlToCheck, 'siteUrl': siteUrl})

if resp.status_code == 200:
  result = resp.json().get('inspectionResult', {}).get('indexStatusResult', {})
  verdict = result.get('verdict', 'UNKNOWN')
  coverage = result.get('coverageState', '')
  lastCrawl = result.get('lastCrawlTime', 'never crawled yet')
  print("Verdict: {}".format(verdict))
  print("Coverage: {}".format(coverage))
  print("Last crawl: {}".format(lastCrawl))
  if verdict == 'PASS':
    print('*'*50);print("Already indexed — pushing again below is optional.")
  else:
    print('*'*50);print("Not indexed yet — run Step 2 below to push it.")
else:
  print("Error Code: {}".format(resp.status_code));print('*'*50);
  print(resp.json());
print('*'*50);

In [ ]:
#@title Step 2 — Push For Indexing (run this if Step 1 said not indexed)
siteURL = "" #@param {type:"string"}
#@markdown Leave blank to reuse the URL you checked in Step 1 above.
requestType = "URL_UPDATED" #@param ["URL_UPDATED", "URL_DELETED"]
if not siteURL:
  siteURL = urlToCheck
print("RESULT:");print('*'*50);print("URL and Update Request Type Configured!");print('*'*50);

response = http.post(ENDPOINT, json={'url': siteURL, 'type': requestType})

if response.status_code == 200:
  print("Successfully Done!");print('*'*50);
else:
  print("Error Code: {}".format(response.status_code));print('*'*50);
  print(response.json());
  print("Visit Here For More: https://developers.google.com/search/apis/indexing-api/v3/core-errors#api-errors");
  print('*'*50);